# AI-Scientist on Claude Code — Colab GPU runner

This notebook is the **compute backend** for `aisci.exec --backend colab`.
It borrows a Colab GPU to run *experiment code only* (model building, training,
evaluation). It does **not** act as an LLM — ideation, writing, and review stay with
the Claude Code agent on your machine.

**How it works:** it mounts your Google Drive and watches
`My Drive/aisci-colab/jobs/`. For each job your local `aisci.exec` submits, it runs
the script on the GPU and writes results back to `.../results/<job_id>/`, which sync
to your Mac via Google Drive for Desktop. It also writes a `RUNNER_ALIVE` heartbeat
so the pipeline can tell a runner is up before routing heavy nodes here.

**Setup (once):**
1. Runtime → Change runtime type → GPU.
2. Run all cells. Approve the Drive mount.
3. Leave the last cell running. On your Mac, set `AISCI_COLAB_SYNC` to the local
   mirror of `My Drive/aisci-colab`, then run experiments with `--backend colab`.

**Safety:** this VM is a throwaway Google sandbox. Only the experiment `code/` is
shipped here — never secrets. Treat it as an untrusted remote.

In [ ]:
# GPU sanity check
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'no nvidia-smi')
try:
    import torch
    print('torch', torch.__version__, 'cuda', torch.cuda.is_available(),
          torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
except Exception as e:
    print('torch not imported:', e)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
SYNC_ROOT = Path('/content/drive/MyDrive/aisci-colab')  # must match local AISCI_COLAB_SYNC
(SYNC_ROOT / 'jobs').mkdir(parents=True, exist_ok=True)
(SYNC_ROOT / 'results').mkdir(parents=True, exist_ok=True)
print('watching', SYNC_ROOT / 'jobs')

In [ ]:
import json, os, sys, time, shutil, subprocess, traceback
from pathlib import Path

JOBS = SYNC_ROOT / 'jobs'
RESULTS = SYNC_ROOT / 'results'
POLL = 10  # seconds


def _find_metric(text):
    """Mirror aisci.exec: last stdout JSON line containing a 'metric' key."""
    found = None
    for line in text.splitlines():
        line = line.strip()
        if line.startswith('{') and line.endswith('}'):
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            if isinstance(obj, dict) and 'metric' in obj:
                found = obj
    return found


def run_job(job_dir):
    job = json.loads((job_dir / 'job.json').read_text())
    node = job['node']
    script = job['script']
    timeout = int(job.get('timeout', 1800))
    seed = job.get('seed')

    work = Path('/content/_aisci') / job['job_id']
    if work.exists():
        shutil.rmtree(work)
    work.mkdir(parents=True)
    shutil.copytree(job_dir / 'code', work / 'code')
    for d in ('experiment_results', 'plots', 'logs'):
        (work / d).mkdir(exist_ok=True)

    env = dict(os.environ)
    if seed is not None:
        env['AISCI_SEED'] = str(seed)

    t0 = time.time()
    timed_out = False
    try:
        proc = subprocess.run([sys.executable, str(work / script)], cwd=str(work),
                              env=env, capture_output=True, text=True, timeout=timeout)
        rc, out, err = proc.returncode, proc.stdout, proc.stderr
    except subprocess.TimeoutExpired as e:
        timed_out = True
        rc = -1
        out = e.stdout or ''
        err = (e.stderr or '') + f'\n[colab_runner] TIMEOUT after {timeout}s'
    dur = time.time() - t0

    (work / 'logs' / f'{node}.log').write_text(
        f'$ {sys.executable} {work / script}\n# cwd={work}\n# rc={rc} dur={dur:.1f}s (colab GPU)\n'
        '\n----- STDOUT -----\n' + (out or '') + '\n----- STDERR -----\n' + (err or ''))

    metric = _find_metric(out or '')
    rp = work / 'experiment_results' / f'{node}.json'
    if metric is None and rp.exists():
        try:
            metric = json.loads(rp.read_text())
        except json.JSONDecodeError:
            metric = None

    is_buggy = rc != 0 or timed_out or metric is None
    record = {
        'node': node, 'ok': not is_buggy, 'is_buggy': is_buggy, 'returncode': rc,
        'timed_out': timed_out, 'duration_s': round(dur, 1), 'metric': metric,
        'seed': seed, 'log': f'experiment/logs/{node}.log',
        'stderr_tail': (err or '')[-800:], 'backend': 'colab',
    }
    (work / 'experiment_results' / f'{node}.json').write_text(json.dumps(record, indent=2))

    res = RESULTS / job['job_id']
    if res.exists():
        shutil.rmtree(res)
    (res / 'logs').mkdir(parents=True)
    (res / 'experiment_results').mkdir(parents=True)
    (res / 'plots').mkdir(parents=True)
    shutil.copy(work / 'logs' / f'{node}.log', res / 'logs' / f'{node}.log')
    for f in (work / 'experiment_results').glob('*'):
        if f.is_file():
            shutil.copy(f, res / 'experiment_results' / f.name)
    for f in (work / 'plots').glob('*'):
        if f.is_file():
            shutil.copy(f, res / 'plots' / f.name)
    (res / 'DONE').write_text(str(time.time()))  # written last
    print(f"[done] {job['job_id']} ok={not is_buggy} dur={dur:.1f}s")


print('runner up; watching', JOBS, '— leave this cell running')
while True:
    try:
        (SYNC_ROOT / 'RUNNER_ALIVE').write_text(str(time.time()))  # heartbeat
        for ready in sorted(JOBS.glob('*/READY')):
            job_dir = ready.parent
            if (job_dir / 'PROCESSED').exists():
                continue
            (job_dir / 'PROCESSED').write_text(str(time.time()))  # claim it once
            print('[run]', job_dir.name)
            try:
                run_job(job_dir)
            except Exception:
                traceback.print_exc()
    except Exception:
        traceback.print_exc()
    time.sleep(POLL)